In [ ]:
import httpx
import pandas as pd
import re

def extrair_dou_vitoria_final():
    url = "https://www.in.gov.br/consulta/-/buscar/dou"
    params = {
        'q': '"ciencia de dados"',
        's': 'todos',
        'exactDate': 'mes',
        'sortType': '0',
        'delta': '100'
    }
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
        'X-Requested-With': 'XMLHttpRequest'
    }

    print("Iniciando extração por varredura de padrões (Regex)...")

    try:
        with httpx.Client(timeout=30.0) as client:
            response = client.get(url, params=params, headers=headers)
            texto = response.text

            # Extração usando padrões de texto (sem depender de json.loads)
            # Buscamos o que está entre "title":" e "
            titulos = re.findall(r'"title":"(.*?)"', texto)
            urls = re.findall(r'"urlTitle":"(.*?)"', texto)
            datas = re.findall(r'"pubDate":"(.*?)"', texto)
            orgaos = re.findall(r'"hierarchyList":\["(.*?)"\]', texto)

            print(f"Padrões localizados: {len(titulos)} títulos.")

            dados = []
            # Alinhamos as listas pelo tamanho dos títulos encontrados
            for i in range(len(titulos)):
                dados.append({
                    'Data': datas[i] if i < len(datas) else "N/A",
                    'Titulo': titulos[i].encode().decode('unicode_escape') if "\\" in titulos[i] else titulos[i],
                    'Orgao': orgaos[i] if i < len(orgaos) else "N/A",
                    'Link': f"https://www.in.gov.br/web/dou/-/{urls[i]}" if i < len(urls) else "N/A"
                })

            if dados:
                df = pd.DataFrame(dados)
                # Limpeza simples de caracteres de escape
                df['Titulo'] = df['Titulo'].str.replace(r'\\', '', regex=True)
                
                df.to_csv("dou_extraido_forca_bruta.csv", index=False, encoding='utf-8-sig')
                print(f"--- SUCESSO TOTAL ---")
                print(f"Arquivo 'dou_extraido_forca_bruta.csv' gerado com {len(dados)} linhas.")
                return df.head()
            else:
                print("Nenhum padrão encontrado no texto. O site pode ter mudado a estrutura.")

    except Exception as e:
        print(f"Erro inesperado: {e}")

extrair_dou_vitoria_final()

Iniciando extração por varredura de padrões (Regex)...
Padrões localizados: 20 títulos.
--- SUCESSO TOTAL ---
Arquivo 'dou_extraido_forca_bruta.csv' gerado com 20 linhas.


,Data,Titulo,Orgao,Link
0,12/02/2026,EDITAL PROGEPE Nº 11 DE 10 DE FEVEREIRO DE 2026,"Ministério da Educação"",""Universidade Federal ...",https://www.in.gov.br/web/dou/-/edital-progepe...
1,12/02/2026,"Portaria UFPR nº 110, de 11 de Fevereiro de 2026","Ministério da Educação"",""Universidade Federal ...",https://www.in.gov.br/web/dou/-/portaria-ufpr-...
2,11/02/2026,"EDITAL 22/GAB/REI/IFPI, de 10 de fevereiro de ...","Ministério da Educação"",""Instituto Federal de ...",https://www.in.gov.br/web/dou/-/edital-22/gab/...
3,11/02/2026,RESULTADO De CHAMADA PÚBLICA IPEA/PIPA Nº 58/2025,"Ministério do Planejamento e Orçamento"",""Insti...",https://www.in.gov.br/web/dou/-/resultado-de-c...
4,11/02/2026,EDITAL Nº 5 DE 10 DE FEVEREIRO DE 2026,"Ministério da Educação"",""Universidade Federal ...",https://www.in.gov.br/web/dou/-/edital-n-5-de-...


In [ ]:
extrair_dou_vitoria_final

In [9]:
import httpx
import pandas as pd
import re
import time

def extrair_concursos_ultima_semana(termos):
    url = "https://www.in.gov.br/consulta/-/buscar/dou"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
        'X-Requested-With': 'XMLHttpRequest'
    }

    todos_os_resultados = []

    for termo in termos:
        print(f"Buscando Concursos (Semana) para: {termo}...")
        
        params = {
            'q': f'"{termo}"',
            's': 'todos',
            'exactDate': 'semana',  # <--- ALTERADO PARA ÚLTIMA SEMANA
            'sortType': '0',
            'delta': '100',
            'artType': 'Edital de Concurso Público'  # <--- TIPO CONCURSO PÚBLICO
        }

        try:
            with httpx.Client(timeout=30.0) as client:
                response = client.get(url, params=params, headers=headers)
                texto = response.text

                # Extração via Regex (Marreta vitoriosa)
                titulos = re.findall(r'"title":"(.*?)"', texto)
                urls = re.findall(r'"urlTitle":"(.*?)"', texto)
                datas = re.findall(r'"pubDate":"(.*?)"', texto)
                tipos = re.findall(r'"artType":"(.*?)"', texto)

                for i in range(len(titulos)):
                    # Decodificação de caracteres especiais (acentos)
                    titulo_limpo = titulos[i].encode().decode('unicode_escape') if "\\" in titulos[i] else titulos[i]
                    titulo_limpo = titulo_limpo.replace('\\', '')

                    todos_os_resultados.append({
                        'Data': datas[i] if i < len(datas) else "N/A",
                        'Tipo': tipos[i] if i < len(tipos) else "Edital de Concurso Público",
                        'Titulo': titulo_limpo,
                        'Link': f"https://www.in.gov.br/web/dou/-/{urls[i]}" if i < len(urls) else "N/A"
                    })
                
                time.sleep(1.5) # Pausa de segurança

        except Exception as e:
            print(f"Erro ao processar termo '{termo}': {e}")

    if todos_os_resultados:
        # Criação do DataFrame e Deduplicação
        df = pd.DataFrame(todos_os_resultados)
        
        total_bruto = len(df)
        # Remove duplicados com base no Link único da matéria
        df = df.drop_duplicates(subset=['Link'], keep='first')
        
        print(f"\n--- RESUMO DA SEMANA ---")
        print(f"Total de editais encontrados: {total_bruto}")
        print(f"Editais repetidos removidos: {total_bruto - len(df)}")
        print(f"Total final de concursos únicos: {len(df)}")

        # Salva o arquivo final
        nome_arquivo = "concursos_dou_ultima_semana.csv"
        df.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')
        print(f"Arquivo '{nome_arquivo}' gerado com sucesso!")
        
        return df
    else:
        print("\nNenhum concurso público encontrado para estes termos nos últimos 7 dias.")


In [10]:

# Execução
termos_concurso = ["ciencia de dados",
                 "inteligencia artificial", 
                 "machine learning",
                 "aprendizado de maquina",
                 "sql",
                 "cientista de dados",
                 "inteligencia da informacao",
                 "analise de dados",
                 "analista de dados",
                 "data science",
                 "data analyst",
                 "Aprendizagem de Máquina",
                 "data analytics",
                 ]

# Execução
extrair_concursos_ultima_semana(termos_concurso)

Buscando Concursos (Semana) para: ciencia de dados...
Erro ao processar termo 'ciencia de dados': [Errno 2] No such file or directory
Buscando Concursos (Semana) para: inteligencia artificial...
Erro ao processar termo 'inteligencia artificial': [Errno 2] No such file or directory
Buscando Concursos (Semana) para: machine learning...
Erro ao processar termo 'machine learning': [Errno 2] No such file or directory
Buscando Concursos (Semana) para: aprendizado de maquina...
Erro ao processar termo 'aprendizado de maquina': [Errno 2] No such file or directory
Buscando Concursos (Semana) para: sql...
Erro ao processar termo 'sql': [Errno 2] No such file or directory
Buscando Concursos (Semana) para: cientista de dados...
Erro ao processar termo 'cientista de dados': [Errno 2] No such file or directory
Buscando Concursos (Semana) para: inteligencia da informacao...
Erro ao processar termo 'inteligencia da informacao': [Errno 2] No such file or directory
Buscando Concursos (Semana) para: anal

In [2]:
import httpx
import pandas as pd
import re
import time
import urllib3

# Silencia os avisos de conexão insegura por estarmos pulando o SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def extrair_concursos_semanal_final(termos):
    url = "https://www.in.gov.br/consulta/-/buscar/dou"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
        'X-Requested-With': 'XMLHttpRequest'
    }

    todos_os_resultados = []

    # O verify=False é o que resolve o [Errno 2] no seu ambiente
    with httpx.Client(timeout=40.0, verify=False, follow_redirects=True) as client:
        for termo in termos:
            print(f"Buscando: {termo}...")
            
            params = {
                'q': f'"{termo}"',
                's': 'todos',
                'exactDate': 'semana',
                'sortType': '0',
                'delta': '100',
                'artType': 'Edital de Concurso Público'
            }

            try:
                response = client.get(url, params=params, headers=headers)
                texto = response.text

                # Extração por Regex
                titulos = re.findall(r'"title":"(.*?)"', texto)
                urls = re.findall(r'"urlTitle":"(.*?)"', texto)
                datas = re.findall(r'"pubDate":"(.*?)"', texto)

                if not titulos:
                    print(f"  -> 0 resultados.")
                    continue

                for i in range(len(titulos)):
                    # Decodificação manual para evitar quebras de biblioteca unicode
                    t_limpo = titulos[i].replace('\\u00ed', 'í').replace('\\u00e1', 'á').replace('\\u00f3', 'ó')
                    t_limpo = t_limpo.replace('\\u00e7', 'ç').replace('\\u00e3', 'ã').replace('\\u00ea', 'ê').replace('\\', '')

                    todos_os_resultados.append({
                        'Data': datas[i] if i < len(datas) else "N/A",
                        'Termo_Original': termo,
                        'Titulo': t_limpo,
                        'Link': f"https://www.in.gov.br/web/dou/-/{urls[i]}" if i < len(urls) else "N/A"
                    })
                
                print(f"  -> {len(titulos)} editais encontrados.")
                time.sleep(1) # Delay para evitar bloqueio por excesso de requisições

            except Exception as e:
                print(f"  -> Erro ao processar '{termo}': {e}")

    if todos_os_resultados:
        df = pd.DataFrame(todos_os_resultados)
        
        # Agrupamento: Junta as palavras-chave que apontam para o mesmo link
        df_consolidado = df.groupby(['Link', 'Data', 'Titulo'])['Termo_Original'].apply(lambda x: ', '.join(sorted(set(x)))).reset_index()
        
        # Reordenar colunas para o CSV ficar organizado
        df_consolidado = df_consolidado[['Data', 'Termo_Original', 'Titulo', 'Link']]
        df_consolidado.columns = ['Data', 'Palavras_Chave', 'Titulo', 'Link']

        # Ordenar pela data mais recente
        df_consolidado = df_consolidado.sort_values(by='Data', ascending=False)

        output_name = "radar_concursos_dados_semanal.csv"
        df_consolidado.to_csv(output_name, index=False, encoding='utf-8-sig')
        
        print(f"\n--- PROCESSO CONCLUÍDO ---")
        print(f"Total de concursos únicos: {len(df_consolidado)}")
        print(f"Arquivo salvo como: {output_name}")
        return df_consolidado
    else:
        print("\nNenhum concurso encontrado para os termos informados nos últimos 7 dias.")

# Sua lista de termos
termos_concurso = [
    "ciencia de dados", "inteligencia artificial", "machine learning",
    "aprendizado de maquina", "sql", "cientista de dados",
    "inteligencia da informacao", "analise de dados", "analista de dados",
    "data science", "data analyst", "Aprendizagem de Máquina", "data analytics"
]

# Execução do radar
extrair_concursos_semanal_final(termos_concurso)

Buscando: ciencia de dados...
  -> 2 editais encontrados.
Buscando: inteligencia artificial...
  -> 3 editais encontrados.
Buscando: machine learning...
  -> 0 resultados.
Buscando: aprendizado de maquina...
  -> 1 editais encontrados.
Buscando: sql...
  -> 3 editais encontrados.
Buscando: cientista de dados...
  -> 0 resultados.
Buscando: inteligencia da informacao...
  -> 0 resultados.
Buscando: analise de dados...
  -> 2 editais encontrados.
Buscando: analista de dados...
  -> 0 resultados.
Buscando: data science...
  -> 0 resultados.
Buscando: data analyst...
  -> 0 resultados.
Buscando: Aprendizagem de Máquina...
  -> 0 resultados.
Buscando: data analytics...
  -> 0 resultados.

--- PROCESSO CONCLUÍDO ---
Total de concursos únicos: 6
Arquivo salvo como: radar_concursos_dados_semanal.csv


,Data,Palavras_Chave,Titulo,Link
2,13/02/2026,aprendizado de maquina,"EDITAL Nº 1, de 12 DE FEVEREIRO DE 2026",https://www.in.gov.br/web/dou/-/edital-n-1-de-...
3,13/02/2026,"analise de dados, inteligencia artificial, sql","EDITAL Nº 2, DE 12 DE FEVEREIRO DE 2026",https://www.in.gov.br/web/dou/-/edital-n-2-de-...
0,11/02/2026,"analise de dados, ciencia de dados, inteligenc...","EDITAL 22/GAB/REI/IFPI, de 10 de fevereiro de ...",https://www.in.gov.br/web/dou/-/edital-22/gab/...
1,11/02/2026,inteligencia artificial,EDITAL DE ABERTURA DE 10 DE FEVEREIRO DE 2026 ...,https://www.in.gov.br/web/dou/-/edital-de-aber...
4,11/02/2026,sql,EDITAL Nº 3/2026,https://www.in.gov.br/web/dou/-/edital-n-3/202...
5,11/02/2026,ciencia de dados,EDITAL Nº 5 DE 10 DE FEVEREIRO DE 2026,https://www.in.gov.br/web/dou/-/edital-n-5-de-...
